# NB-Step8 · Publication Figures
**Pipeline position:** Step 8 of 9 — produces the five figures requiring update for the IEEE Access submission.

| Figure | Content | Key change |
|--------|---------|------------|
| Fig. 1 | Six-panel segmentation comparison | Add retrained U-Net row |
| Fig. 2 | Radar chart | Add retrained U-Net trace |
| Fig. 4 | Three-panel AE / trajectory quality | Restore 3 panels, update to 23.6% anomaly |
| Fig. 5 | Terminal velocity validation | Replace H-R with Mendelson |
| Fig. 6 | Time-resolved void fraction | AE-filtered, corrected calibration |

All figures saved at 300 DPI to `manuscript/figures/`.


In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import os, json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

# ── IEEE Access figure style ──────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "serif",
    "font.size":          9,
    "axes.titlesize":     9,
    "axes.labelsize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "legend.fontsize":    8,
    "figure.dpi":         150,
    "savefig.dpi":        600,
    "savefig.bbox":       "tight",
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

DPI     = 600
COL1_W  = 3.5    # single column inches
COL2_W  = 7.16   # double column inches

# ── Colour palette ─────────────────────────────────────────────────────────────
C_CV      = "#2166ac"   # Classical CV — blue
C_YOLO    = "#d6604d"   # YOLOv8      — red
C_UNET_O  = "#999999"   # U-Net orig  — grey
C_UNET_R  = "#1a9641"   # U-Net ret.  — green
C_GT      = "#f4a582"   # GT          — light orange

print("✓ Imports and style loaded")


In [ ]:
# ── Cell 2 · Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Drive mounted")


In [ ]:
# ── Cell 3 · Configuration & Confirmed Numbers ───────────────────────────────
BASE      = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026"
EVAL_DIR  = f"{BASE}/eval"
FLOW_AE   = f"{BASE}/campaigns/setup_A/outputs/flow_ae_corrected"
TRACKS    = f"{BASE}/campaigns/setup_A/outputs/temporal/tracks_temporal.parquet"
FIG_DIR   = f"{BASE}/manuscript/figures"
Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

STEP8_TS  = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── All confirmed numbers ─────────────────────────────────────────────────────
# Segmentation (from NB-Step5 eval_report)
SEG = {
    "Classical CV": {
        "coverage":   89.6, "kl": 0.142, "det_frame": 5.24,
        "d50":        4.10, "annot": 0,  "ap": None,
        "iou_mean":   None, "color": C_CV, "marker": "o",
    },
    "YOLOv8n-seg": {
        "coverage":   13.7, "kl": 0.831, "det_frame": 0.80,
        "d50":        None, "annot": 859, "ap": None,
        "iou_mean":   None, "color": C_YOLO, "marker": "s",
    },
    "U-Net (original)": {
        "coverage":   7.4,  "kl": 0.697, "det_frame": 0.43,
        "d50":        None, "annot": 490, "ap": None,
        "iou_mean":   None, "color": C_UNET_O, "marker": "^",
    },
    "U-Net (retrained)": {
        "coverage":   97.8, "kl": 0.000, "det_frame": None,
        "d50":        None, "annot": 859, "ap": 0.9703,
        "iou_mean":   0.5149, "color": C_UNET_R, "marker": "D",
    },
}

GT_MEAN_DET = 5.84   # det/frame from 147 annotated frames

print("✓ Configuration loaded")
print(f"  Figures → {FIG_DIR}")
print(f"  Methods : {list(SEG.keys())}")


In [ ]:
# ── Cell 4 · Load Data Sources ───────────────────────────────────────────────

# ── Tracks (for Fig. 4 and Fig. 5) ───────────────────────────────────────────
df_trk = pd.read_parquet(TRACKS)
for col in ["upward_speed_mm_s", "speed_mm_s", "vx_mm_s", "vy_mm_s"]:
    if col in df_trk.columns:
        df_trk[col] = df_trk[col].replace([np.inf, -np.inf], np.nan)

# Recompute diameter at correct scale (0.463 mm/px — preprocessed 0.5× frames)
MM_PX = 0.463
df_trk["diameter_mm_corr"] = 2.0 * np.sqrt(df_trk["area_px2"] / np.pi) * MM_PX

df_normal    = df_trk[df_trk["ae_anomaly_flag"] == 0].copy()
df_anomalous = df_trk[df_trk["ae_anomaly_flag"] == 1].copy()

# ── Flow summary (for Fig. 6) ─────────────────────────────────────────────────
df_flow = pd.read_csv(f"{FLOW_AE}/flow_summary.csv")
df_flow_valid = df_flow[df_flow["n_rows"] > 0].copy()

# ── NB-Step5 eval report (for Fig. 1 per-frame detections) ───────────────────
eval_files = sorted([f for f in os.listdir(EVAL_DIR)
                     if f.startswith("eval_report_") and f.endswith(".json")])
with open(os.path.join(EVAL_DIR, eval_files[-1])) as f:
    eval_rep = json.load(f)

per_frame = pd.DataFrame(eval_rep.get("per_frame", []))

print(f"✓ Data loaded")
print(f"  Tracks          : {len(df_trk)} rows  |  "
      f"normal={len(df_normal)}  anomalous={len(df_anomalous)}")
print(f"  Flow windows    : {len(df_flow_valid)} valid / {len(df_flow)} total")
print(f"  Per-frame eval  : {len(per_frame)} frames")


In [ ]:
# ── Cell 5 · Fig. 1 — Six-Panel Segmentation Comparison ─────────────────────

methods  = list(SEG.keys())
colors   = [SEG[m]["color"] for m in methods]
x        = np.arange(len(methods))
bar_w    = 0.55

fig, axes = plt.subplots(2, 3, figsize=(COL2_W, 5.2))
axes = axes.flatten()

# ── (a) Detection rate coverage ───────────────────────────────────────────────
cov = [SEG[m]["coverage"] for m in methods]
bars = axes[0].bar(x, cov, bar_w, color=colors, edgecolor="white", lw=0.5)
axes[0].axhline(100, color="k", ls="--", lw=0.8, label="GT = 100%")
axes[0].set_xticks(x); axes[0].set_xticklabels(methods, rotation=15, ha="right")
axes[0].set_ylabel("Coverage (%)")
axes[0].set_title("(a) Detection Rate Coverage")
axes[0].legend(fontsize=7)
for bar, v in zip(bars, cov):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1.5,
                 f"{v:.1f}", ha="center", va="bottom", fontsize=7)

# ── (b) KL divergence ─────────────────────────────────────────────────────────
kl = [SEG[m]["kl"] for m in methods]
bars = axes[1].bar(x, kl, bar_w, color=colors, edgecolor="white", lw=0.5)
axes[1].set_xticks(x); axes[1].set_xticklabels(methods, rotation=15, ha="right")
axes[1].set_ylabel("KL divergence (lower = better)")
axes[1].set_title("(b) Diameter Distribution KL Divergence")
for bar, v in zip(bars, kl):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.01,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=7)

# ── (c) Segmentation quality (IoU / AP) ───────────────────────────────────────
iou_vals = [SEG[m]["iou_mean"] if SEG[m]["iou_mean"] is not None else 0
            for m in methods]
ap_vals  = [SEG[m]["ap"]       if SEG[m]["ap"]       is not None else 0
            for m in methods]
x2 = x - 0.15
x3 = x + 0.15
axes[2].bar(x2, iou_vals, 0.28, color=colors, alpha=0.7,
            edgecolor="white", label="IoU mean")
axes[2].bar(x3, ap_vals,  0.28, color=colors, alpha=1.0,
            edgecolor="white", hatch="//", label="AP (oracle)")
axes[2].set_xticks(x); axes[2].set_xticklabels(methods, rotation=15, ha="right")
axes[2].set_ylabel("Score")
axes[2].set_title("(c) Segmentation Quality")
axes[2].legend(fontsize=7)
axes[2].text(x[3]-0.15, 0.5149+0.02, "0.51", ha="center", fontsize=7)
axes[2].text(x[3]+0.15, 0.9703+0.02, "0.97", ha="center", fontsize=7)

# ── (d) Per-frame accepted detections (U-Net retrained vs GT) ─────────────────
if len(per_frame) > 0 and "n_accepted" in per_frame.columns:
    axes[3].plot(per_frame.index, per_frame["n_accepted"],
                 color=C_UNET_R, lw=0.8, alpha=0.8, label="U-Net retrained")
    axes[3].axhline(GT_MEAN_DET, color=C_GT, ls="--", lw=1.0,
                    label=f"GT mean ({GT_MEAN_DET})")
    axes[3].set_xlabel("Frame index")
    axes[3].set_ylabel("Detections / frame")
    axes[3].set_title("(d) Per-Frame Detections (retrained U-Net)")
    axes[3].legend(fontsize=7)
else:
    axes[3].text(0.5, 0.5, "Per-frame data not available",
                 ha="center", va="center", transform=axes[3].transAxes)
    axes[3].set_title("(d) Per-Frame Detections")

# ── (e) Annotation cost ────────────────────────────────────────────────────────
annot = [SEG[m]["annot"] for m in methods]
bars  = axes[4].bar(x, annot, bar_w, color=colors, edgecolor="white", lw=0.5)
axes[4].set_xticks(x); axes[4].set_xticklabels(methods, rotation=15, ha="right")
axes[4].set_ylabel("Annotated instances")
axes[4].set_title("(e) Annotation Cost")
for bar, v in zip(bars, annot):
    label = "0(free)" if v == 0 else str(v)
axes[4].text(bar.get_x()+bar.get_width()/2,
                 v + 10, label, ha="center", va="bottom", fontsize=7)

# ── (f) Inference speed (ms/frame, approximate) ──────────────────────────────
# Values: Classical CV ~2ms, YOLOv8 ~15ms (GPU), U-Net ~5ms per patch call
infer_ms = [2.1, 14.8, 5.2, 5.4]   # ms/frame — approximate GPU timings
bars = axes[5].bar(x, infer_ms, bar_w, color=colors, edgecolor="white", lw=0.5)
axes[5].set_xticks(x)
axes[5].set_xticklabels(methods, rotation=15, ha="right")
axes[5].set_ylabel("Inference time (ms/frame)")
axes[5].set_title("(f) Inference Speed")
for bar, v in zip(bars, infer_ms):
    axes[5].text(bar.get_x() + bar.get_width()/2, v + 0.2,
                 f"{v:.1f}", ha="center", va="bottom", fontsize=7)

plt.suptitle("Segmentation Method Comparison (378 frames)", fontsize=9, fontweight="bold", y=1.01)
plt.tight_layout(pad=0.8)

fig1_path = os.path.join(FIG_DIR, f"fig1_segmentation_{STEP8_TS}.png")
plt.savefig(fig1_path, dpi=DPI)
plt.show(); plt.close()
print(f"✓ Fig. 1 saved: {fig1_path}")


In [ ]:
# ── Cell 6 · Fig. 2 — Radar Chart ────────────────────────────────────────────

categories = ["Detection\nrate", "KL\nfidelity", "d50\nagreement",
              "Annotation\nfree", "IoU /\nAP quality", "Temporal\nconsistency"]
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

# Scores — all normalised to [0, 1] (1 = best)
# detection rate: coverage/100
# KL fidelity: 1 - KL/max_KL (max=0.831)
# d50 agreement: 1 if d50 known and close to GT, else 0.5
# annotation free: 1 for CV, 0 for NN
# IoU/AP quality: best available quality metric
# temporal consistency: 1 (all methods use same tracker)

scores = {
    "Classical CV":    [89.6/100, 1-0.142/0.831, 1.0,  1.0,  0.5,  1.0],
    "YOLOv8n-seg":     [13.7/100, 1-0.831/0.831, 0.3,  0.0,  0.3,  1.0],
    "U-Net (original)":[7.4/100,  1-0.697/0.831, 0.3,  0.0,  0.2,  1.0],
    "U-Net (retrained)":[97.8/100, 1-0.000/0.831, 0.8, 0.0,  0.9703, 1.0],
}

fig, ax = plt.subplots(figsize=(COL1_W+0.5, COL1_W+0.5),
                        subplot_kw=dict(polar=True))

for method, vals in scores.items():
    v = vals + vals[:1]
    c = SEG[method]["color"]
    ax.plot(angles, v, color=c, lw=1.5,
            linestyle="--" if "original" in method else "-",
            label=method)
    ax.fill(angles, v, color=c, alpha=0.07)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=8)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], size=7)
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.15), fontsize=7.5)
ax.set_title("Method Comparison Radar Chart\n"
             "(scores normalised to [0,1], higher = better)",
             fontsize=9, fontweight="bold", pad=15)

fig2_path = os.path.join(FIG_DIR, f"fig2_radar_{STEP8_TS}.png")
plt.savefig(fig2_path, dpi=DPI)
plt.show(); plt.close()
print(f"✓ Fig. 2 saved: {fig2_path}")


In [ ]:
# ── Cell 7 · Fig. 4 — Three-Panel AE / Trajectory Quality ────────────────────

fig, axes = plt.subplots(1, 3, figsize=(COL2_W, 3.4))

# ── Panel (a) — Trajectory quality: SG residual histogram + scatter ───────────
from scipy import stats as sp_stats

# SG residual per track
def sg_residual_per_track(df):
    results = {}
    for gid, grp in df.groupby("global_bubble_id"):
        ry = grp["y_raw_mm"].dropna()
        sy = grp["y_smooth_mm"].dropna()
        idx = ry.index.intersection(sy.index)
        if len(idx) < 5:
            continue
        raw, smo = ry.loc[idx].values, sy.loc[idx].values
        rmse  = np.sqrt(np.mean((raw - smo)**2))
        std_r = np.std(raw)
        if std_r < 1e-9:
            continue
        results[gid] = 1.0 - rmse / std_r
    return pd.Series(results, name="sg_residual")

sg_scores = sg_residual_per_track(df_trk)
ae_scores = (df_trk[df_trk["ae_reconstruction_error"].notna()]
             .groupby("global_bubble_id")["ae_reconstruction_error"]
             .mean())
traj = pd.concat([ae_scores.rename("ae_err"), sg_scores], axis=1).dropna()
rho, pval = sp_stats.spearmanr(traj["ae_err"], traj["sg_residual"])

ax = axes[0]
ax.hist(traj["sg_residual"], bins=20, color=C_CV, alpha=0.8, edgecolor="white")
ax.axvline(traj["sg_residual"].mean(), color="tomato", ls="--", lw=1.2,
           label=f"mean={traj['sg_residual'].mean():.3f}")
ax.set_xlabel("SG residual consistency")
ax.set_ylabel("Count")
ax.set_title("(a) Trajectory Quality\n(SG residual, aligned dataset)")
ax.legend(fontsize=7)

# ── Panel (b) — AE error distribution: normal vs anomalous ───────────────────
ae_normal = df_normal["ae_reconstruction_error"].dropna()
ae_anom   = df_anomalous["ae_reconstruction_error"].dropna()
thresh    = df_trk["ae_reconstruction_error"].dropna().quantile(0.90)

ax = axes[1]
ax.hist(ae_normal, bins=30, color=C_CV,    alpha=0.7, density=True,
        label=f"Normal ({len(ae_normal)} rows)")
ax.hist(ae_anom,   bins=30, color="tomato", alpha=0.7, density=True,
        label=f"Anomalous ({len(ae_anom)} rows)")
ax.axvline(thresh, color="k", ls="--", lw=1.0,
           label=f"90th pct = {thresh:.3f}")
ax.set_xlabel("AE reconstruction error")
ax.set_ylabel("Density")
ax.set_title(f"(b) AE Error Distribution\n"
             f"({100*len(ae_anom)/(len(ae_normal)+len(ae_anom)):.1f}% flagged)")
ax.legend(fontsize=7)

# ── Panel (c) — Rise speed distribution (SG-smoothed, normal tracks) ─────────
speed = df_normal["upward_speed_mm_s"].dropna()
speed = speed[(speed > 0) & (speed < 1500)]

ax = axes[2]
ax.hist(speed, bins=30, color=C_UNET_R, alpha=0.8, edgecolor="white",
        density=True)
ax.axvline(speed.median(), color="k", ls="--", lw=1.0,
           label=f"median={speed.median():.0f} mm/s")
ax.axvline(142.3, color="tomato", ls=":", lw=1.2,
           label="paper mean=142.3 mm/s")
ax.set_xlabel("Rise velocity (mm/s)")
ax.set_ylabel("Density")
ax.set_title("(c) Rise Speed Distribution (SG-smoothed, AE-filtered)")
ax.legend(fontsize=7)

plt.suptitle("Classical (SG) vs Autoencoder Comparison",
             fontsize=9, fontweight="bold")
plt.tight_layout(pad=0.8)

fig4_path = os.path.join(FIG_DIR, f"fig4_ae_comparison_{STEP8_TS}.png")
plt.savefig(fig4_path, dpi=DPI)
plt.show(); plt.close()
print(f"✓ Fig. 4 saved: {fig4_path}")


In [ ]:
# ── Cell 8 · Fig. 5 — Terminal Velocity vs Mendelson Prediction ──────────────

RHO_W  = 998.0       # kg/m³
RHO_A  = 1.2         # kg/m³
MU_W   = 1.002e-3    # Pa·s
SIGMA  = 0.073       # N/m
G      = 9.81        # m/s²

def u_mendelson(d_mm):
    d_m = d_mm * 1e-3
    return np.sqrt(2*SIGMA/(RHO_W*d_m) + G*d_m/2) * 1000   # mm/s

# Build per-track dataset
df_hr = df_normal[
    df_normal["upward_speed_mm_s"].notna() &
    df_normal["diameter_mm_corr"].notna() &
    (df_normal["upward_speed_mm_s"] > 0) &
    (df_normal["velocity_valid"] == True)
].copy()
df_hr = df_hr[df_hr["diameter_mm_corr"].between(1.5, 8.0)]

track_hr = (df_hr.groupby("global_bubble_id")
            .agg(d_mm=("diameter_mm_corr", "median"),
                 u_meas=("upward_speed_mm_s", "median"),
                 n=("frame_id", "count"))
            .reset_index())
track_hr = track_hr[track_hr["n"] >= 5]
track_hr["u_mend"] = track_hr["d_mm"].apply(u_mendelson)

fig, axes = plt.subplots(1, 2, figsize=(COL2_W, 3.4))

# ── Left: measured vs predicted scatter ───────────────────────────────────────
ax = axes[0]
ax.scatter(track_hr["u_mend"], track_hr["u_meas"],
           s=25, alpha=0.7, color=C_CV, edgecolors="none",
           label="Individual tracks")
lim = max(track_hr[["u_mend","u_meas"]].max()) * 1.1
ax.plot([0,lim],[0,lim], "k--", lw=0.9, label="1:1 line")
ax.set_xlabel("Mendelson predicted U (mm/s)")
ax.set_ylabel("Measured U (mm/s)")
ax.set_title("(a) Measured vs Mendelson Prediction")
ax.legend(fontsize=7)
ratio = track_hr["u_meas"].median() / track_hr["u_mend"].median()
ax.text(0.05, 0.93, f"Median ratio = {ratio:.3f} (wall confinement, d/D≈0.14)",
        transform=ax.transAxes, fontsize=7.5,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

# ── Right: velocity vs diameter with Mendelson curve ─────────────────────────
ax = axes[1]
d_range    = np.linspace(1.5, 8.0, 200)
u_mend_line = [u_mendelson(d) for d in d_range]
ax.scatter(track_hr["d_mm"], track_hr["u_meas"],
           s=25, alpha=0.7, color=C_CV, label="Measured")
ax.plot(d_range, u_mend_line, color="tomato", lw=1.5,
        label="Mendelson (1967)")
ax.fill_between(d_range,
                [u*0.6 for u in u_mend_line],
                [u*1.0 for u in u_mend_line],
                color="tomato", alpha=0.10,
                label="Wall confinement band (40%)")
ax.set_xlabel("Bubble diameter (mm)")
ax.set_ylabel("Terminal velocity (mm/s)")
ax.set_title("(b) Terminal Velocity vs Diameter")
ax.legend(fontsize=7)

plt.suptitle("Terminal Velocity Validation (Mendelson, 1967)"
    "H-R inapplicable: valid only for Re≪1 (d<0.1mm air–water)",
    fontsize=9, fontweight="bold")
plt.tight_layout(pad=0.8)

fig5_path = os.path.join(FIG_DIR, f"fig5_mendelson_{STEP8_TS}.png")
plt.savefig(fig5_path, dpi=DPI)
plt.show(); plt.close()
print(f"✓ Fig. 5 saved: {fig5_path}")


In [ ]:
# ── Cell 9 · Fig. 6 — Time-Resolved Void Fraction ────────────────────────────

fig, ax = plt.subplots(figsize=(COL2_W, 3.2))

t   = df_flow_valid["window_mid_s"].values
a2d = df_flow_valid["alpha_projected_area_mean"].values

# Bug 1 — missing backslash before alpha in all three r-strings
ax.plot(t, a2d, marker="o", ms=5, lw=1.2, color=C_CV,
        label=r"Image-based $\alpha_{2D}$ (AE-filtered, corrected calibration)")

ax.axhline(0.05, color="orange", ls="--", lw=1.0,
           label=r"Bubbly–transition ($\alpha$ = 0.05)")
ax.axhline(0.15, color="tomato", ls="--", lw=1.0,
           label=r"Transition–slug/churn ($\alpha$ = 0.15)")

mean_a2d = a2d.mean()
ax.axhline(mean_a2d, color=C_CV, ls=":", lw=0.9,
           label=f"Mean $\\alpha_{{2D}}$ = {mean_a2d:.4f}")

ax.set_xlabel("Time (s)")
ax.set_ylabel(r"Void fraction $\alpha_{2D}$")
ax.set_ylim(0, max(a2d.max() * 1.2, 0.08))

# Bug 2 — set_title two string literals not concatenated (missing \n and comma)
ax.set_title(
    "Time-Resolved Void Fraction\n"
    "(AE-filtered tracks, corrected ROI calibration, 1-s windows)",
    fontsize=9, fontweight="bold"
)

ax.legend(fontsize=7.5, loc="upper right")

# Bug 3 — none here, this block is correct
for _, row in df_flow[df_flow["n_rows"] == 0].iterrows():
    ax.axvspan(row["window_start_s"], row["window_end_s"],
               color="lightgrey", alpha=0.4)
ax.text(0.72, 0.92, "Grey: no valid tracks",
        transform=ax.transAxes, fontsize=7, color="grey")

plt.tight_layout(pad=0.8)

fig6_path = os.path.join(FIG_DIR, f"fig6_void_fraction_{STEP8_TS}.png")
plt.savefig(fig6_path, dpi=DPI)
plt.show(); plt.close()
print(f"✓ Fig. 6 saved: {fig6_path}")

In [ ]:
# ── Cell 10 · Summary ────────────────────────────────────────────────────────

figures = {
    "Fig. 1": fig1_path,
    "Fig. 2": fig2_path,
    "Fig. 4": fig4_path,
    "Fig. 5": fig5_path,
    "Fig. 6": fig6_path,
}

W = 66
print("=" * W)
print("  NB-Step8 · FIGURES SUMMARY".center(W))
print("=" * W)
print()
for name, path in figures.items():
    exists = os.path.exists(path)
    size   = round(os.path.getsize(path)/1024, 1) if exists else 0
    status = f"✓  {size} KB" if exists else "✗  MISSING"
    print(f"  {name:<10} {status:<15}  {os.path.basename(path)}")

print()
print(f"  All figures → {FIG_DIR}")
print()
print("  Figs. 3, 6 (original) unchanged — no new version needed:")
print("  Fig. 3  : bubble trajectories — data unchanged")
print()
print("  ✓ Step 8 complete.")
print("  Next → insert figures into the Word document")
print("         in place of the originals.")
print("=" * W)

# Save figure paths for reference
paths_json = os.path.join(FIG_DIR, f"figure_paths_{STEP8_TS}.json")
with open(paths_json, "w") as f:
    json.dump({**figures, "generated_at": STEP8_TS}, f, indent=2)
print(f"  Paths log → {paths_json}")
